# Step 1. Upload csv files and transfer them to DuckDB file

Load all csv files into DuctDB tables

In [ ]:
import pandas as pd
import duckdb
import os

# Initialize DuckDB in-memory database
con = duckdb.connect(database=':memory:', read_only=False)

# Define the directory where CSV files are located
csv_dir = '/content/'

# Get a list of all files in the directory
all_files = os.listdir(csv_dir)

# Filter to include only CSV files
csv_files = [f for f in all_files if f.endswith('.csv')]

print(f"Found {len(csv_files)} CSV files in '{csv_dir}'.")

# Process each CSV file
for file_name in csv_files:
    full_path = os.path.join(csv_dir, file_name)
    try:
        # Read the CSV file into a pandas DataFrame, interpreting 'None' as nulls
        df = pd.read_csv(full_path, na_values=['None'])

        # Clean up file name for DuckDB table name (remove extension and replace invalid characters)
        table_name = file_name.rsplit('.', 1)[0].replace(' ', '_').replace('-', '_')

        # Register DataFrame as a DuckDB table
        con.execute(f"CREATE OR REPLACE TABLE {table_name} AS SELECT * FROM df")
        print(f"'{file_name}' loaded into DuckDB table: '{table_name}' with 'None' as nulls.")
    except Exception as e:
        print(f"Error processing '{file_name}': {e}")

print("\nAll available CSVs are now loaded as tables in DuckDB (if successful).")

# Verify the data loading
print("\n--- Verification ---")
print("Tables in DuckDB:")
all_tables = con.execute("SHOW TABLES").fetchall()
for table_tuple in all_tables:
    table_name = table_tuple[0]
    print(f"\nTable: {table_name}")
    print("Schema:")
    print(con.execute(f"DESCRIBE {table_name}").fetchdf())
    print("First 5 rows (including potential nulls):")
    print(con.execute(f"SELECT * FROM {table_name} LIMIT 5").fetchdf())
    print("\n")

Based on results, common columns are `customer_id`, `order_id`, `review_id`, `product_id`, `seller_i`, and `order_item_id`.

Create a SQL view named `main_orders_view` by performing LEFT JOINs on

`olist_orders_dataset`

`olist_customers_dataset`

`olist_order_items_dataset`

`olist_order_payments_dataset`

using `order_id` and `customer_id` as common columns, then verify the view creation and its schema.


In [ ]:
sql_query = """CREATE OR REPLACE VIEW main_orders_view AS
SELECT
    o.*,
    c.customer_unique_id,
    c.customer_zip_code_prefix,
    c.customer_city,
    c.customer_state,
    oi.order_item_id,
    oi.product_id,
    oi.seller_id,
    oi.shipping_limit_date,
    oi.price,
    oi.freight_value,
    op.payment_sequential,
    op.payment_type,
    op.payment_installments,
    op.payment_value
FROM olist_orders_dataset AS o
LEFT JOIN olist_customers_dataset AS c
    ON o.customer_id = c.customer_id
LEFT JOIN olist_order_items_dataset AS oi
    ON o.order_id = oi.order_id
LEFT JOIN olist_order_payments_dataset AS op
    ON o.order_id = op.order_id;
"""

con.execute(sql_query)
print("View 'main_orders_view' created successfully in DuckDB.")

# Verify the view by listing tables/views and describing it
print("\nTables and Views in DuckDB after creation:")
print(con.execute("SHOW TABLES").fetchall())
print("\nSchema of 'main_orders_view':")
print(con.execute("DESCRIBE main_orders_view").fetchall())

**Summary**

The process of creating the `main_orders_view` involved constructing a SQL query that performed `LEFT JOIN` operations on four datasets: `olist_orders_dataset`

`olist_customers_dataset`

`olist_order_items_dataset`

`olist_order_payments_dataset`.


The joins were made using `order_id` and `customer_id` as common columns.

The view's readiness for further analysis was confirmed by successfully executing the creation query, verifying its presence among database objects, and describing its schema, which accurately reflected the joined columns.



**Data Analysis Key Findings**
*   A SQL view named `main_orders_view` was successfully created in the DuckDB database using a `CREATE OR REPLACE VIEW` statement.
*   The `main_orders_view` was confirmed to be present in the database through the `SHOW TABLES` command.
*   The schema of `main_orders_view` was verified using `DESCRIBE main_orders_view`, confirming that all selected columns from the joined tables (`olist_orders_dataset`, `olist_customers_dataset`, `olist_order_items_dataset`, and `olist_order_payments_dataset`) were correctly included, indicating successful `LEFT JOIN` operations.

**Insights or Next Steps**
*   The `main_orders_view` is now available and properly structured, providing a unified dataset ready for comprehensive order-related analysis.



In [ ]:
# Calculate basic statistics from the main_orders_view
stats_query = """
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT order_id) AS unique_orders,
    COUNT(DISTINCT customer_unique_id) AS unique_customers
FROM main_orders_view;
"""

# Fetch the results into a pandas DataFrame
df_stats = con.execute(stats_query).fetchdf()

# Print the results
print("--- Main Orders View Statistics ---")
print(df_stats.to_string(index=False))

# Calculate and print order-to-customer ratio
ratio = df_stats['unique_orders'][0] / df_stats['unique_customers'][0]
print(f"\nAverage orders per unique customer: {ratio:.2f}")

In [ ]:
# check total records in each table
for table_tuple in all_tables:
    # Extract the name from the tuple (usually the first element)
    table_name = table_tuple[0]

    print(f"\n{'='*30}")
    print(f"Table: {table_name}")
    print(f"{'='*30}")

    print("\nSchema:")
    print(con.execute(f"DESCRIBE {table_name}").fetchdf())

    print("\nFirst 5 rows (including potential nulls):")
    # Added the closing parenthesis and .fetchdf() to complete the logic
    print(con.execute(f"SELECT * FROM {table_name} LIMIT 5").fetchdf())

# check total record of each table in all_tables
print("\n--- Total Records in Each Table ---")
# Calculate the max length for pretty alignment (optional but nice)
max_len = max(len(t[0]) for t in all_tables)

for table_tuple in all_tables:
    table_name = table_tuple[0]
    # Fetch the count
    total_records = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]

    # Use f-string padding to align the counts
    print(f"{table_name:<{max_len + 2}} : {total_records:,}")

Export the in-memory DuckDB database to a persistent file and download it to the local machine.


In [ ]:
import os
from google.colab import files

# 1. Configuration
db_file = 'olist_ecommerce.db'
alias = 'persistent_db'

# 2. Cleanup: Handle existing attachments or old files
try:
    existing_dbs = con.execute("SELECT database_name FROM duckdb_databases()").fetchall()
    if any(db[0] == alias for db in existing_dbs):
        print(f"Detaching existing {alias}...")
        con.execute(f"DETACH {alias}")
except Exception:
    pass # If we can't even check databases, we'll just try to move on

if os.path.exists(db_file):
    os.remove(db_file)
    print(f"Removed old {db_file} to start fresh.")

# 3. Attach the new database file
con.execute(f"ATTACH '{db_file}' AS {alias}")

# 4. Identify objects to migrate (Legacy-Compatible Version)
# Get Tables
tables = con.execute("SELECT table_name FROM duckdb_tables() WHERE schema_name = 'main'").fetchall()
# Get Views
views = con.execute("SELECT view_name FROM duckdb_views() WHERE schema_name = 'main'").fetchall()

print(f"Found {len(tables)} tables and {len(views)} views to export.")

# 5. Migration Loop - Tables
for (table_name,) in tables:
    try:
        con.execute(f"CREATE TABLE {alias}.{table_name} AS SELECT * FROM main.{table_name}")
        print(f"  [Table] {table_name} copied.")
    except Exception as e:
        print(f"  [Error] Could not copy table {table_name}: {e}")

# 6. Migration Loop - Views
for (view_name,) in views:
    try:
        # Get the SQL definition
        view_sql = con.execute(f"SELECT sql FROM duckdb_views() WHERE view_name = '{view_name}'").fetchone()[0]
        # Recreate view (splitting at 'AS' to get the query part)
        query_part = view_sql.split('AS', 1)[1]
        con.execute(f"CREATE VIEW {alias}.{view_name} AS {query_part}")
        print(f"  [View]  {view_name} recreated.")
    except Exception as e:
        print(f"  [Error] Could not recreate view {view_name}: {e}")

# 7. Finalize and Download
con.execute(f"DETACH {alias}")
print(f"\n✅ Database successfully exported to {db_file}")

files.download(db_file)

# Step 2. Create a Star Schema in the Same DuctDB Data File

In [ ]:
import duckdb
import os
from google.colab import files

db_file = 'olist_ecommerce.db'

# Check if the database file already exists in the Colab environment
if os.path.exists(db_file):
    print(f"'{db_file}' found. Connecting to existing database.")
    con = duckdb.connect(database=db_file, read_only=False)
else:
    print(f"'{db_file}' not found. Please upload the DuckDB file from your local machine.")
    print("If you previously exported it, it would have been downloaded to your computer.")
    uploaded = files.upload() # This will prompt user to upload.

    if db_file not in uploaded:
        print(f"Error: '{db_file}' was not uploaded. Please ensure you upload the correct file.")
        # Raise an error to stop execution if the required file isn't uploaded
        raise FileNotFoundError(f"'{db_file}' was not uploaded. Please restart the cell and upload the file.")
    else:
        print(f"'{db_file}' uploaded successfully. Connecting to database.")
        con = duckdb.connect(database=db_file, read_only=False)

print(f"Successfully connected to DuckDB database: {db_file}")

# Verify tables in the loaded database
print("\nTables in the loaded DuckDB database:")
print(con.execute("SHOW TABLES").fetchdf())

In [ ]:
# create a 'Star Schema' called 'analytics' if not exist in db_file

# Create the 'analytics' schema if it doesn't already exist
con.execute("CREATE SCHEMA IF NOT EXISTS analytics;")
print("Schema 'analytics' created or already exists.")

# Verify the creation by listing schemas using the correct DuckDB system view
print("\nSchemas in DuckDB:")
print(con.execute("SELECT * FROM duckdb_schemas();").fetchdf())

Further steps:

1.  **Analyze Data for Star Schema Design**:
    *   Examine the schema of `main_orders_view` to identify core order-related information.
    *   Examine `olist_customers_dataset`, `olist_products_dataset`, `product_category_name_translation`, `olist_sellers_dataset`, and `olist_geolocation_dataset` to gather attributes for respective dimensions.
    *   Define the columns and their data types for the following dimension tables: `dim_customer`, `dim_product`, `dim_seller`, `dim_geolocation`, and `dim_time`.
    *   Define the columns and their data types for the `fact_orders` table, including foreign keys to the dimension tables and measurable facts.

2.  **Create Dimension Tables (DDL)**:
    *   Generate and execute SQL DDL statements to create the following tables in the `analytics` schema, including surrogate keys for each:
        *   `analytics.dim_customer`
        *   `analytics.dim_product`
        *   `analytics.dim_seller`
        *   `analytics.dim_geolocation`
        *   `analytics.dim_time`

3.  **Populate Dimension Tables (DML)**:
    *   Generate and execute SQL INSERT statements to populate each dimension table by selecting distinct values and relevant attributes from the source tables and `main_orders_view`. Assign surrogate keys during this process.

4.  **Create Fact Table (DDL)**:
    *   Generate and execute SQL DDL statements to create `analytics.fact_orders`, including foreign key columns referencing the surrogate keys of the dimension tables and all identified measures (facts).

5.  **Populate Fact Table (DML)**:
    *   Generate and execute SQL INSERT statements to populate `analytics.fact_orders`, joining `main_orders_view` with the newly created dimension tables to correctly map natural keys to surrogate keys and insert the factual data.

6.  **Verify Star Schema Creation**:
    *   Verify the successful creation and population of all dimension and fact tables by listing tables within the `analytics` schema.
    *   Display the schema and a sample of data for each created table to confirm correct structure and content.

7.  **Final Task**:
    *   Summarize the successfully created star schema, including the names of the dimension and fact tables, and confirm their readiness for analytical queries.


Examine the schemas of `main_orders_view`, `olist_products_dataset`, `product_category_name_translation`, `olist_sellers_dataset`, and `olist_geolocation_dataset` to identify appropriate attributes for each dimension and fact table.


In [ ]:
print("\n--- Schema of main_orders_view ---")
print(con.execute("DESCRIBE main_orders_view").fetchdf())

print("\n--- Schema of olist_products_dataset ---")
print(con.execute("DESCRIBE olist_products_dataset").fetchdf())

print("\n--- Schema of product_category_name_translation ---")
print(con.execute("DESCRIBE product_category_name_translation").fetchdf())

print("\n--- Schema of olist_sellers_dataset ---")
print(con.execute("DESCRIBE olist_sellers_dataset").fetchdf())

print("\n--- Schema of olist_geolocation_dataset ---")
print(con.execute("DESCRIBE olist_geolocation_dataset").fetchdf())


Based on the schema analysis of `main_orders_view`, `olist_products_dataset`, `product_category_name_translation`, `olist_sellers_dataset`, and `olist_geolocation_dataset`, the following dimension and fact tables are proposed for the `analytics` schema:

### 1. Dimension Table: `analytics.dim_customer`
This table will store unique customer information.
- `customer_pk` (BIGINT, PRIMARY KEY, AUTOINCREMENT): Surrogate key for the customer dimension.
- `customer_id` (VARCHAR): Natural key from `olist_customers_dataset` and `main_orders_view`.
- `customer_unique_id` (VARCHAR): Unique identifier for a customer from `main_orders_view`.
- `customer_zip_code_prefix` (BIGINT): Customer's zip code prefix from `main_orders_view`.
- `customer_city` (VARCHAR): Customer's city from `main_orders_view`.
- `customer_state` (VARCHAR): Customer's state from `main_orders_view`.

### 2. Dimension Table: `analytics.dim_product`
This table will store product details.
- `product_pk` (BIGINT, PRIMARY KEY, AUTOINCREMENT): Surrogate key for the product dimension.
- `product_id` (VARCHAR): Natural key from `olist_products_dataset` and `main_orders_view`.
- `product_category_name` (VARCHAR): Original product category name from `olist_products_dataset`.
- `product_category_name_english` (VARCHAR): Translated product category name from `product_category_name_translation`.
- `product_name_lenght` (DOUBLE): Product name length from `olist_products_dataset`.
- `product_description_lenght` (DOUBLE): Product description length from `olist_products_dataset`.
- `product_photos_qty` (DOUBLE): Number of product photos from `olist_products_dataset`.
- `product_weight_g` (DOUBLE): Product weight in grams from `olist_products_dataset`.
- `product_length_cm` (DOUBLE): Product length in cm from `olist_products_dataset`.
- `product_height_cm` (DOUBLE): Product height in cm from `olist_products_dataset`.
- `product_width_cm` (DOUBLE): Product width in cm from `olist_products_dataset`.

### 3. Dimension Table: `analytics.dim_seller`
This table will store seller information.
- `seller_pk` (BIGINT, PRIMARY KEY, AUTOINCREMENT): Surrogate key for the seller dimension.
- `seller_id` (VARCHAR): Natural key from `olist_sellers_dataset` and `main_orders_view`.
- `seller_zip_code_prefix` (BIGINT): Seller's zip code prefix from `olist_sellers_dataset`.
- `seller_city` (VARCHAR): Seller's city from `olist_sellers_dataset`.
- `seller_state` (VARCHAR): Seller's state from `olist_sellers_dataset`.

### 4. Dimension Table: `analytics.dim_geolocation`
This table will store geographical data, primarily based on zip code prefixes.
- `geolocation_pk` (BIGINT, PRIMARY KEY, AUTOINCREMENT): Surrogate key for the geolocation dimension.
- `geolocation_zip_code_prefix` (BIGINT): Zip code prefix from `olist_geolocation_dataset`.
- `geolocation_lat` (DOUBLE): Latitude from `olist_geolocation_dataset`.
- `geolocation_lng` (DOUBLE): Longitude from `olist_geolocation_dataset`.
- `geolocation_city` (VARCHAR): City name from `olist_geolocation_dataset`.
- `geolocation_state` (VARCHAR): State name from `olist_geolocation_dataset`.

### 5. Dimension Table: `analytics.dim_time`
This table will store date and time components derived from order timestamps.
- `time_pk` (BIGINT, PRIMARY KEY, AUTOINCREMENT): Surrogate key for the time dimension (YYYYMMDD).
- `full_date` (DATE): The full date.
- `year` (INTEGER): Year component.
- `month` (INTEGER): Month component (1-12).
- `day` (INTEGER): Day component (1-31).
- `quarter` (INTEGER): Quarter of the year (1-4).
- `day_of_week` (INTEGER): Day of the week (1=Sunday, 7=Saturday).
- `day_name` (VARCHAR): Name of the day (e.g., 'Monday').
- `month_name` (VARCHAR): Name of the month (e.g., 'January').
- `week_of_year` (INTEGER): Week number of the year.
- `is_weekend` (BOOLEAN): Flag indicating if the day is a weekend.

### 6. Fact Table: `analytics.fact_orders`
This table will store the core transactional data and measures.
- `order_pk` (BIGINT, PRIMARY KEY, AUTOINCREMENT): Surrogate key for the fact table.
- `order_id` (VARCHAR): Natural key from `main_orders_view`.
- `customer_fk` (BIGINT, FOREIGN KEY): References `analytics.dim_customer.customer_pk`.
- `product_fk` (BIGINT, FOREIGN KEY): References `analytics.dim_product.product_pk`.
- `seller_fk` (BIGINT, FOREIGN KEY): References `analytics.dim_seller.seller_pk`.
- `order_purchase_date_fk` (BIGINT, FOREIGN KEY): References `analytics.dim_time.time_pk` for order purchase date.
- `order_approved_date_fk` (BIGINT, FOREIGN KEY): References `analytics.dim_time.time_pk` for order approved date.
- `order_delivered_carrier_date_fk` (BIGINT, FOREIGN KEY): References `analytics.dim_time.time_pk` for carrier delivery date.
- `order_delivered_customer_date_fk` (BIGINT, FOREIGN KEY): References `analytics.dim_time.time_pk` for customer delivery date.
- `order_estimated_delivery_date_fk` (BIGINT, FOREIGN KEY): References `analytics.dim_time.time_pk` for estimated delivery date.
- `customer_zip_geolocation_fk` (BIGINT, FOREIGN KEY): References `analytics.dim_geolocation.geolocation_pk` for customer location.
- `seller_zip_geolocation_fk` (BIGINT, FOREIGN KEY): References `analytics.dim_geolocation.geolocation_pk` for seller location.
- `order_status` (VARCHAR): Current status of the order.
- `order_item_id` (BIGINT): Identifier for each item within an order (from `main_orders_view`).
- `shipping_limit_date` (VARCHAR): The date when the shipping limit was met (from `main_orders_view`).
- `price` (DOUBLE): Price of the product item (from `main_orders_view`).
- `freight_value` (DOUBLE): Freight value for the item (from `main_orders_view`).
- `payment_sequential` (BIGINT): Sequential number of a payment means for a given order (from `main_orders_view`).
- `payment_type` (VARCHAR): Type of payment used (from `main_orders_view`).
- `payment_installments` (BIGINT): Number of installments for the payment (from `main_orders_view`).
- `payment_value` (DOUBLE): Total value of the payment (from `main_orders_view`).


In [ ]:
print("Creating dimension tables in 'analytics' schema...")

# 1. Create analytics.dim_customer
con.execute("""CREATE OR REPLACE TABLE analytics.dim_customer (
    customer_pk BIGINT PRIMARY KEY,
    customer_id VARCHAR,
    customer_unique_id VARCHAR,
    customer_zip_code_prefix BIGINT,
    customer_city VARCHAR,
    customer_state VARCHAR
);
""")
print("Created analytics.dim_customer.")

# 2. Create analytics.dim_product
con.execute("""CREATE OR REPLACE TABLE analytics.dim_product (
    product_pk BIGINT PRIMARY KEY,
    product_id VARCHAR,
    product_category_name VARCHAR,
    product_category_name_english VARCHAR,
    product_name_lenght DOUBLE,
    product_description_lenght DOUBLE,
    product_photos_qty DOUBLE,
    product_weight_g DOUBLE,
    product_length_cm DOUBLE,
    product_height_cm DOUBLE,
    product_width_cm DOUBLE
);
""")
print("Created analytics.dim_product.")

# 3. Create analytics.dim_seller
con.execute("""CREATE OR REPLACE TABLE analytics.dim_seller (
    seller_pk BIGINT PRIMARY KEY,
    seller_id VARCHAR,
    seller_zip_code_prefix BIGINT,
    seller_city VARCHAR,
    seller_state VARCHAR
);
""")
print("Created analytics.dim_seller.")

# 4. Create analytics.dim_geolocation
con.execute("""CREATE OR REPLACE TABLE analytics.dim_geolocation (
    geolocation_pk BIGINT PRIMARY KEY,
    geolocation_zip_code_prefix BIGINT,
    geolocation_lat DOUBLE,
    geolocation_lng DOUBLE,
    geolocation_city VARCHAR,
    geolocation_state VARCHAR
);
""")
print("Created analytics.dim_geolocation.")

# 5. Create analytics.dim_time
# Note: This dim_time table creation will be more complex as it needs to generate dates.
# For now, create a basic structure. Population will be done later.
con.execute("""CREATE OR REPLACE TABLE analytics.dim_time (
    time_pk INTEGER PRIMARY KEY,
    full_date DATE,
    year INTEGER,
    month INTEGER,
    day INTEGER,
    quarter INTEGER,
    day_of_week INTEGER,
    day_name VARCHAR,
    month_name VARCHAR,
    week_of_year INTEGER,
    is_weekend BOOLEAN
);
""")
print("Created analytics.dim_time.")

print("All dimension tables DDL executed.")

In [ ]:
# check dim_customer, populate it



Drop and recreate the dimension tables (`analytics.dim_customer`, `analytics.dim_product`, `analytics.dim_seller`, `analytics.dim_geolocation`, `analytics.dim_time`) using `GENERATED ALWAYS AS IDENTITY` for their primary keys, then populate `analytics.dim_customer` with distinct customer information from `main_orders_view`, and finally verify its schema, first 5 rows, and total record count.

## Drop and Recreate Dimension Tables

### Subtask:
Drop the existing dimension tables (`analytics.dim_customer`, `analytics.dim_product`, `analytics.dim_seller`, `analytics.dim_geolocation`, `analytics.dim_time`) and then recreate them using `GENERATED ALWAYS AS IDENTITY` for their primary keys to ensure proper auto-incrementing functionality in DuckDB.


**Reasoning**:
The subtask requires dropping existing dimension tables and then recreating them with `GENERATED ALWAYS AS IDENTITY` for their primary keys. This step will ensure that the primary keys auto-increment correctly. I will generate the SQL `DROP TABLE` statements followed by the `CREATE TABLE` statements for each dimension table.



In [ ]:
print("Dropping and recreating dimension tables in 'analytics' schema...")

# Drop existing tables
con.execute("DROP TABLE IF EXISTS analytics.dim_customer;")
con.execute("DROP TABLE IF EXISTS analytics.dim_product;")
con.execute("DROP TABLE IF EXISTS analytics.dim_seller;")
con.execute("DROP TABLE IF EXISTS analytics.dim_geolocation;")
con.execute("DROP TABLE IF EXISTS analytics.dim_time;")
print("Existing dimension tables dropped.")

# 1. Recreate analytics.dim_customer
con.execute("""CREATE OR REPLACE TABLE analytics.dim_customer (
    customer_pk BIGINT PRIMARY KEY, -- Removed GENERATED ALWAYS AS IDENTITY
    customer_id VARCHAR,
    customer_unique_id VARCHAR,
    customer_zip_code_prefix BIGINT,
    customer_city VARCHAR,
    customer_state VARCHAR
);
""")
print("Recreated analytics.dim_customer with BIGINT PRIMARY KEY.")

# 2. Recreate analytics.dim_product
con.execute("""CREATE OR REPLACE TABLE analytics.dim_product (
    product_pk BIGINT PRIMARY KEY, -- Removed GENERATED ALWAYS AS IDENTITY
    product_id VARCHAR,
    product_category_name VARCHAR,
    product_category_name_english VARCHAR,
    product_name_lenght DOUBLE,
    product_description_lenght DOUBLE,
    product_photos_qty DOUBLE,
    product_weight_g DOUBLE,
    product_length_cm DOUBLE,
    product_height_cm DOUBLE,
    product_width_cm DOUBLE
);
""")
print("Recreated analytics.dim_product with BIGINT PRIMARY KEY.")

# 3. Recreate analytics.dim_seller
con.execute("""CREATE OR REPLACE TABLE analytics.dim_seller (
    seller_pk BIGINT PRIMARY KEY, -- Removed GENERATED ALWAYS AS IDENTITY
    seller_id VARCHAR,
    seller_zip_code_prefix BIGINT,
    seller_city VARCHAR,
    seller_state VARCHAR
);
""")
print("Recreated analytics.dim_seller with BIGINT PRIMARY KEY.")

# 4. Recreate analytics.dim_geolocation
con.execute("""CREATE OR REPLACE TABLE analytics.dim_geolocation (
    geolocation_pk BIGINT PRIMARY KEY, -- Removed GENERATED ALWAYS AS IDENTITY
    geolocation_zip_code_prefix BIGINT,
    geolocation_lat DOUBLE,
    geolocation_lng DOUBLE,
    geolocation_city VARCHAR,
    geolocation_state VARCHAR
);
""")
print("Recreated analytics.dim_geolocation with BIGINT PRIMARY KEY.")

# 5. Recreate analytics.dim_time
con.execute("""CREATE OR REPLACE TABLE analytics.dim_time (
    time_pk BIGINT PRIMARY KEY, -- Removed GENERATED ALWAYS AS IDENTITY
    full_date DATE,
    year INTEGER,
    month INTEGER,
    day INTEGER,
    quarter INTEGER,
    day_of_week INTEGER,
    day_name VARCHAR,
    month_name VARCHAR,
    week_of_year INTEGER,
    is_weekend BOOLEAN
);
""")
print("Recreated analytics.dim_time with BIGINT PRIMARY KEY.")

print("All dimension tables dropped and recreated with BIGINT primary keys.")

**Reasoning**:
Now that the dimension tables are created with `BIGINT PRIMARY KEY`, the next step is to populate the `analytics.dim_customer` table with distinct customer information from `main_orders_view`. I will assign a `customer_pk` using `ROW_NUMBER()` and then verify the schema, first 5 rows, and total record count of the populated table.



In [ ]:
print("Populating analytics.dim_customer...")

con.execute("""INSERT INTO analytics.dim_customer (customer_pk, customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_state)
SELECT
    ROW_NUMBER() OVER (ORDER BY customer_id) AS customer_pk,
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix,
    customer_city,
    customer_state
FROM (
    SELECT DISTINCT
        customer_id,
        customer_unique_id,
        customer_zip_code_prefix,
        customer_city,
        customer_state
    FROM main_orders_view
) AS distinct_customers;
""")
print("analytics.dim_customer populated successfully.")

# Verify analytics.dim_customer
print("\n--- Verification of analytics.dim_customer ---")
print("Schema:")
print(con.execute("DESCRIBE analytics.dim_customer").fetchdf())

print("\nFirst 5 rows:")
print(con.execute("SELECT * FROM analytics.dim_customer LIMIT 5").fetchdf())

print("\nTotal record count:")
print(con.execute("SELECT COUNT(*) FROM analytics.dim_customer").fetchdf())

## Populate dim_product

### Subtask:
Populate the `analytics.dim_product` table with distinct product information from `olist_products_dataset` and `product_category_name_translation`.


**Reasoning**:
The subtask requires populating the `analytics.dim_product` table with distinct product information from `olist_products_dataset` and `product_category_name_translation`, assigning a `product_pk` using `ROW_NUMBER()`, and then verifying the table's schema, first 5 rows, and total record count.



In [ ]:
print("Populating analytics.dim_product...")

con.execute("""INSERT INTO analytics.dim_product (
    product_pk,
    product_id,
    product_category_name,
    product_category_name_english,
    product_name_lenght,
    product_description_lenght,
    product_photos_qty,
    product_weight_g,
    product_length_cm,
    product_height_cm,
    product_width_cm
)
SELECT
    ROW_NUMBER() OVER (ORDER BY op.product_id) AS product_pk,
    op.product_id,
    op.product_category_name,
    pcnt.product_category_name_english,
    op.product_name_lenght,
    op.product_description_lenght,
    op.product_photos_qty,
    op.product_weight_g,
    op.product_length_cm,
    op.product_height_cm,
    op.product_width_cm
FROM (
    SELECT DISTINCT
        product_id,
        product_category_name,
        product_name_lenght,
        product_description_lenght,
        product_photos_qty,
        product_weight_g,
        product_length_cm,
        product_height_cm,
        product_width_cm
    FROM olist_products_dataset
) AS op
LEFT JOIN product_category_name_translation AS pcnt
    ON op.product_category_name = pcnt.product_category_name;
""")
print("analytics.dim_product populated successfully.")

# Verify analytics.dim_product
print("\n--- Verification of analytics.dim_product ---")
print("Schema:")
print(con.execute("DESCRIBE analytics.dim_product").fetchdf())

print("\nFirst 5 rows:")
print(con.execute("SELECT * FROM analytics.dim_product LIMIT 5").fetchdf())

print("\nTotal record count:")
print(con.execute("SELECT COUNT(*) FROM analytics.dim_product").fetchdf())

## Populate dim_seller

### Subtask:
Populate the `analytics.dim_seller` table with distinct seller information from `olist_sellers_dataset`.


**Reasoning**:
The subtask requires populating the `analytics.dim_seller` table with distinct seller information from `olist_sellers_dataset`, assigning a `seller_pk` using `ROW_NUMBER()`, and then verifying the table's schema, first 5 rows, and total record count.



In [ ]:
print("Populating analytics.dim_seller...")

con.execute("""INSERT INTO analytics.dim_seller (
    seller_pk,
    seller_id,
    seller_zip_code_prefix,
    seller_city,
    seller_state
)
SELECT
    ROW_NUMBER() OVER (ORDER BY seller_id) AS seller_pk,
    seller_id,
    seller_zip_code_prefix,
    seller_city,
    seller_state
FROM (
    SELECT DISTINCT
        seller_id,
        seller_zip_code_prefix,
        seller_city,
        seller_state
    FROM olist_sellers_dataset
) AS distinct_sellers;
""")
print("analytics.dim_seller populated successfully.")

# Verify analytics.dim_seller
print("\n--- Verification of analytics.dim_seller ---")
print("Schema:")
print(con.execute("DESCRIBE analytics.dim_seller").fetchdf())

print("\nFirst 5 rows:")
print(con.execute("SELECT * FROM analytics.dim_seller LIMIT 5").fetchdf())

print("\nTotal record count:")
print(con.execute("SELECT COUNT(*) FROM analytics.dim_seller").fetchdf())

## Populate dim_geolocation

### Subtask:
Populate the `analytics.dim_geolocation` table with distinct geographical information from `olist_geolocation_dataset`.


**Reasoning**:
The subtask requires populating the `analytics.dim_geolocation` table with distinct geographical information from `olist_geolocation_dataset`. I will use `ROW_NUMBER()` to assign a unique `geolocation_pk` and then verify the schema, first 5 rows, and total record count of the populated table.

**Reasoning**:
Now that the reasoning for populating `analytics.dim_geolocation` is provided, I will generate the SQL query to populate the `analytics.dim_geolocation` table and verify its contents as instructed.



In [ ]:
print("Populating analytics.dim_geolocation...")

con.execute("""INSERT INTO analytics.dim_geolocation (
    geolocation_pk,
    geolocation_zip_code_prefix,
    geolocation_lat,
    geolocation_lng,
    geolocation_city,
    geolocation_state
)
SELECT
    ROW_NUMBER() OVER (ORDER BY geolocation_zip_code_prefix, geolocation_lat, geolocation_lng) AS geolocation_pk,
    geolocation_zip_code_prefix,
    geolocation_lat,
    geolocation_lng,
    geolocation_city,
    geolocation_state
FROM (
    SELECT DISTINCT
        geolocation_zip_code_prefix,
        geolocation_lat,
        geolocation_lng,
        geolocation_city,
        geolocation_state
    FROM olist_geolocation_dataset
) AS distinct_geolocations;
""")
print("analytics.dim_geolocation populated successfully.")

# Verify analytics.dim_geolocation
print("\n--- Verification of analytics.dim_geolocation ---")
print("Schema:")
print(con.execute("DESCRIBE analytics.dim_geolocation").fetchdf())

print("\nFirst 5 rows:")
print(con.execute("SELECT * FROM analytics.dim_geolocation LIMIT 5").fetchdf())

print("\nTotal record count:")
print(con.execute("SELECT COUNT(*) FROM analytics.dim_geolocation").fetchdf())

## Populate dim_time

### Subtask:
Populate the `analytics.dim_time` table with a comprehensive range of dates and their corresponding time attributes.


**Reasoning**:
I need to determine the minimum and maximum order purchase dates from the `main_orders_view` to define the range for generating the time dimension. This will provide the start and end points for `generate_series`.



In [ ]:
print("Determining date range for dim_time...")

date_range_query = """SELECT
    MIN(CAST(order_purchase_timestamp AS DATE)) AS min_date,
    MAX(CAST(order_purchase_timestamp AS DATE)) AS max_date
FROM main_orders_view;"""

date_range = con.execute(date_range_query).fetchdf()

min_date = date_range['min_date'][0]
max_date = date_range['max_date'][0]

print(f"Minimum order purchase date: {min_date}")
print(f"Maximum order purchase date: {max_date}")

**Reasoning**:
Now that the date range is determined, I will populate the `analytics.dim_time` table using a `generate_series` function to create dates between `min_date` and `max_date`, extracting all specified time attributes and assigning a `time_pk` in YYYYMMDD format. Finally, I will verify the table's schema, first 5 rows, and total record count.



In [ ]:
print("Populating analytics.dim_time...")

con.execute("DELETE FROM analytics.dim_time;")
print("Existing data in analytics.dim_time deleted.")

# Insert a default 'Unknown' date entry
con.execute("""
INSERT INTO analytics.dim_time (time_pk, full_date, year, month, day, quarter, day_of_week, day_name, month_name, week_of_year, is_weekend)
VALUES (0, NULL, NULL, NULL, NULL, NULL, NULL, 'Unknown', 'Unknown', NULL, NULL);
""")
print("Inserted 'Unknown' date entry into analytics.dim_time.")

con.execute(f"""INSERT INTO analytics.dim_time (
    time_pk,
    full_date,
    year,
    month,
    day,
    quarter,
    day_of_week,
    day_name,
    month_name,
    week_of_year,
    is_weekend
)
SELECT
    CAST(STRFTIME(date_series, '%Y%m%d') AS BIGINT) AS time_pk,
    date_series AS full_date,
    EXTRACT(YEAR FROM date_series) AS year,
    EXTRACT(MONTH FROM date_series) AS month,
    EXTRACT(DAY FROM date_series) AS day,
    EXTRACT(QUARTER FROM date_series) AS quarter,
    (EXTRACT(DOW FROM date_series) + 1) AS day_of_week, -- Adjust to 1=Sunday, 7=Saturday
    STRFTIME(date_series, '%A') AS day_name, -- Use %A for full weekday name (e.g., 'Sunday')
    STRFTIME(date_series, '%B') AS month_name, -- Use %B for full month name (e.g., 'January')
    EXTRACT(WEEK FROM date_series) AS week_of_year,
    CASE WHEN EXTRACT(DOW FROM date_series) IN (0, 6) THEN TRUE ELSE FALSE END AS is_weekend
FROM (
    SELECT UNNEST(GENERATE_SERIES('{min_date}'::DATE, '{max_date}'::DATE, INTERVAL 1 DAY)) AS date_series
) AS dates;
""")
print("analytics.dim_time populated successfully with generated dates.")

# Verify analytics.dim_time
print("\n--- Verification of analytics.dim_time ---")
print("Schema:")
print(con.execute("DESCRIBE analytics.dim_time").fetchdf())

print("\nFirst 5 rows (including Unknown if present):")
print(con.execute("SELECT * FROM analytics.dim_time ORDER BY time_pk LIMIT 5").fetchdf())

print("\nTotal record count:")
print(con.execute("SELECT COUNT(*) FROM analytics.dim_time").fetchdf())

# Task
Adjust the SQL query for populating `analytics.dim_time` to correctly cast `date_series` to a `DATE` type before using `STRFTIME`, resolving the `BinderException`. Then, re-execute the population query and verify the schema, first 5 rows, and total record count for `analytics.dim_time`.

## Resolve dim_time Population Error

### Subtask:
Adjust the SQL query for populating `analytics.dim_time` to correctly cast `date_series` to a `DATE` type before using `STRFTIME`, resolving the `BinderException`. Then, re-execute the population query and verify the schema, first 5 rows, and total record count for `analytics.dim_time`.


## Create Fact Table

### Subtask:
Generate and execute the SQL DDL statement to create the `analytics.fact_orders` table, including all specified columns, foreign keys referencing the surrogate keys of the dimension tables, and measures.


**Reasoning**:
I need to generate the SQL DDL statement to create the `analytics.fact_orders` table as specified in the instructions, including primary key, foreign keys, and all other columns. Then, I will execute this statement and print a confirmation.



In [ ]:
# check if analytics.fact_orders is created, clean all data
result = con.execute("SELECT COUNT(*) FROM analytics.fact_orders;").fetchone()
print(f"Total records in fact_orders: {result[0]}")


In [ ]:
print("Creating analytics.fact_orders table...")

con.execute("""CREATE OR REPLACE TABLE analytics.fact_orders (
    order_pk BIGINT PRIMARY KEY,
    order_id VARCHAR,
    customer_fk BIGINT,
    product_fk BIGINT,
    seller_fk BIGINT,
    order_purchase_date_fk BIGINT,
    order_approved_date_fk BIGINT,
    order_delivered_carrier_date_fk BIGINT,
    order_delivered_customer_date_fk BIGINT,
    order_estimated_delivery_date_fk BIGINT,
    customer_zip_geolocation_fk BIGINT,
    seller_zip_geolocation_fk BIGINT,
    order_status VARCHAR,
    order_item_id BIGINT,
    shipping_limit_date VARCHAR,
    price DOUBLE,
    freight_value DOUBLE,
    payment_sequential BIGINT,
    payment_type VARCHAR,
    payment_installments BIGINT,
    payment_value DOUBLE,
    FOREIGN KEY (customer_fk) REFERENCES analytics.dim_customer(customer_pk),
    FOREIGN KEY (product_fk) REFERENCES analytics.dim_product(product_pk),
    FOREIGN KEY (seller_fk) REFERENCES analytics.dim_seller(seller_pk),
    FOREIGN KEY (order_purchase_date_fk) REFERENCES analytics.dim_time(time_pk),
    FOREIGN KEY (order_approved_date_fk) REFERENCES analytics.dim_time(time_pk),
    FOREIGN KEY (order_delivered_carrier_date_fk) REFERENCES analytics.dim_time(time_pk),
    FOREIGN KEY (order_delivered_customer_date_fk) REFERENCES analytics.dim_time(time_pk),
    FOREIGN KEY (order_estimated_delivery_date_fk) REFERENCES analytics.dim_time(time_pk),
    FOREIGN KEY (customer_zip_geolocation_fk) REFERENCES analytics.dim_geolocation(geolocation_pk),
    FOREIGN KEY (seller_zip_geolocation_fk) REFERENCES analytics.dim_geolocation(geolocation_pk)
);
""")

print("analytics.fact_orders table created successfully.")

## Populate Fact Table (DML)

### Subtask:
Generate and execute SQL INSERT statements to populate `analytics.fact_orders`, joining `main_orders_view` with the newly created dimension tables to correctly map natural keys to surrogate keys and insert the factual data.

**Reasoning**:
Now that the fact table `analytics.fact_orders` has been created, I will generate the SQL INSERT statement to populate it by joining `main_orders_view` with the dimension tables to map natural keys to surrogate keys. This is the next logical step in building the star schema.



In [ ]:
print("Populating analytics.fact_orders...")

con.execute("""INSERT INTO analytics.fact_orders (
    order_pk,
    order_id,
    customer_fk,
    product_fk,
    seller_fk,
    order_purchase_date_fk,
    order_approved_date_fk,
    order_delivered_carrier_date_fk,
    order_delivered_customer_date_fk,
    order_estimated_delivery_date_fk,
    customer_zip_geolocation_fk,
    seller_zip_geolocation_fk,
    order_status,
    order_item_id,
    shipping_limit_date,
    price,
    freight_value,
    payment_sequential,
    payment_type,
    payment_installments,
    payment_value
)
SELECT
    ROW_NUMBER() OVER (ORDER BY mov.order_id, mov.order_item_id) AS order_pk, /* Ensure unique PK for each order_item */
    mov.order_id,
    dc.customer_pk,
    dp.product_pk,
    ds.seller_pk,
    CAST(STRFTIME(CAST(mov.order_purchase_timestamp AS DATE), '%Y%m%d') AS BIGINT) AS order_purchase_date_fk,
    CAST(STRFTIME(CAST(mov.order_approved_at AS DATE), '%Y%m%d') AS BIGINT) AS order_approved_date_fk,
    CAST(STRFTIME(CAST(mov.order_delivered_carrier_date AS DATE), '%Y%m%d') AS BIGINT) AS order_delivered_carrier_date_fk,
    CAST(STRFTIME(CAST(mov.order_delivered_customer_date AS DATE), '%Y%m%d') AS BIGINT) AS order_delivered_customer_date_fk,
    CAST(STRFTIME(CAST(mov.order_estimated_delivery_date AS DATE), '%Y%m%d') AS BIGINT) AS order_estimated_delivery_date_fk,
    COALESCE(dc_geo.geolocation_pk, -1) AS customer_zip_geolocation_fk, -- Use -1 or a default if not found
    COALESCE(ds_geo.geolocation_pk, -1) AS seller_zip_geolocation_fk, -- Use -1 or a default if not found
    mov.order_status,
    mov.order_item_id,
    mov.shipping_limit_date,
    mov.price,
    mov.freight_value,
    mov.payment_sequential,
    mov.payment_type,
    mov.payment_installments,
    mov.payment_value
FROM main_orders_view AS mov
LEFT JOIN analytics.dim_customer AS dc
    ON mov.customer_id = dc.customer_id AND mov.customer_unique_id = dc.customer_unique_id
LEFT JOIN analytics.dim_product AS dp
    ON mov.product_id = dp.product_id
LEFT JOIN analytics.dim_seller AS ds
    ON mov.seller_id = ds.seller_id
LEFT JOIN analytics.dim_geolocation AS dc_geo
    ON mov.customer_zip_code_prefix = dc_geo.geolocation_zip_code_prefix
LEFT JOIN analytics.dim_geolocation AS ds_geo
    ON ds.seller_zip_code_prefix = ds_geo.geolocation_zip_code_prefix;
""")
print("analytics.fact_orders populated successfully.")

# Verify analytics.fact_orders
print("\n--- Verification of analytics.fact_orders ---")
print("Schema:")
print(con.execute("DESCRIBE analytics.fact_orders").fetchdf())

print("\nFirst 5 rows:")
print(con.execute("SELECT * FROM analytics.fact_orders LIMIT 5").fetchdf())

print("\nTotal record count:")
print(con.execute("SELECT COUNT(*) FROM analytics.fact_orders").fetchdf())